In [ ]:
import pandas as pd
import numpy as np
import re
import networkx as nx
from matplotlib.lines import Line2D
import os
import cstarpy.inference
import cstarpy.integration
from cstarpy.preprocessing import PerturbationMagnitude
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import pylab as plt
import openpyxl

In [ ]:
# 4. Align to Szabo reference vector (raw) 

REF_COL = "log2FoldChange"

common_genes  = pivot.columns.intersection(cd8_szabo.index)
pivot_aligned = pivot[common_genes]
szabo_aligned = cd8_szabo.loc[common_genes, REF_COL].values

#  5. DPD per compound = pivot @ szabo (no normalisation)

dpd_vec = pivot_aligned.values @ szabo_aligned

dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                       for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)

#  6. Add targets and mechanism 

targets = pd.read_csv(COMPOUND_FILE)[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")

dpd_df = dpd_df.merge(targets, on="compound_name", how="left")

#  7. Save 
dpd_df.to_csv("dpd_sum_per_compound_raw_btla.csv", index=False)
print("\n✓ Saved dpd_sum_per_compound_raw_btla.csv")
